# P3-B VerSe Multi-Window Dataset Processing

This notebook creates a data-level Mask2Former ablation for VerSe 2D. The baseline dataset stores one bone-window CT slice and later feeds it as repeated grayscale channels. P3-B keeps the same slice selection and label export, but saves each image as a 3-channel CT multi-window PNG:

- **R**: soft-tissue window `[-160, 240]`
- **G**: bone window `[-500, 1300]`
- **B**: dense-bone window `[200, 1800]`

The model source should remain baseline Mask2Former. This notebook only changes the data representation.

## 1. User Configuration

In [ ]:
import os
import sys
import glob
import json
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# Resolve the repository root whether the notebook is launched from repo root or this folder.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UTILS_DIR = PROJECT_ROOT / 'utils'
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from data_utilities import *

# Raw VerSe data root. Expected structure: {split}/rawdata/sub-*/ and {split}/derivatives/sub-*/.
INPUT_ROOT = PROJECT_ROOT / 'data'

# Multi-window processed data output.
OUTPUT_ROOT = PROJECT_ROOT / 'dataset_verse_2d_multiwindow'

SPLITS = ['train', 'val', 'test']
EXPORT_STYLES = ['ade20k', 'coco']
SELECTED_TASKS_BY_STYLE = {
    'ade20k': ['semantic'],
    'coco': ['instance'],
}

# Same slice filtering policy as the baseline process_dataset notebook.
THRESHOLD_PERCENT = 0.20
MIN_PIXELS = 500
FORCE_REPROCESS = False

# CT windows for multi-window input.
WINDOWS = {
    'soft_tissue': (-160, 240),
    'bone': (-500, 1300),
    'dense_bone': (200, 1800),
}

V_NAMES = ['C1','C2','C3','C4','C5','C6','C7','T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12','L1','L2','L3','L4','L5','L6','Sacrum','Cocc','T13']


def map_semantic_class(inst_id):
    if 1 <= inst_id <= 7:
        return 1  # Cervical
    if (8 <= inst_id <= 19) or (inst_id == 28):
        return 2  # Thoracic
    if 20 <= inst_id <= 25:
        return 3  # Lumbar
    return 0

print('Input root:', INPUT_ROOT.resolve())
print('Output root:', OUTPUT_ROOT.resolve())
print('Export styles:', EXPORT_STYLES)

## 2. Windowing Utilities

In [ ]:
def window_image(img, min_hu, max_hu):
    """Convert one HU slice to uint8 using a CT window."""
    img = np.clip(img, min_hu, max_hu)
    img = (img - min_hu) / float(max_hu - min_hu) * 255.0
    return img.astype(np.uint8)


def multi_window_image(img):
    """Create RGB-like CT input: [soft tissue, bone, dense bone]."""
    soft = window_image(img, *WINDOWS['soft_tissue'])
    bone = window_image(img, *WINDOWS['bone'])
    dense = window_image(img, *WINDOWS['dense_bone'])
    return np.stack([soft, bone, dense], axis=-1).astype(np.uint8)


def get_panoptic_encoding(inst_msk):
    unique_ids = np.unique(inst_msk).astype(int)
    unique_ids = unique_ids[unique_ids > 0]
    pan_img = np.zeros((inst_msk.shape[0], inst_msk.shape[1], 3), dtype=np.uint8)
    segments_info = []
    for uid in unique_ids:
        mask = inst_msk == uid
        area = int(np.sum(mask))
        pan_img[mask, 0] = uid % 256
        pan_img[mask, 1] = uid // 256
        pan_img[mask, 2] = 0
        y, x = np.where(mask)
        bbox = [float(np.min(x)), float(np.min(y)), float(np.max(x) - np.min(x) + 1), float(np.max(y) - np.min(y) + 1)]
        segments_info.append({
            'id': int(uid),
            'category_id': int(uid),
            'isthing': True,
            'iscrowd': 0,
            'area': area,
            'bbox': bbox,
            'bbox_mode': 1,
        })
    return pan_img, segments_info

## 3. Quick Visual Check

This optional cell compares the baseline repeated bone-window input with the multi-window input on the same sample slice. It does not write data.

In [ ]:
sample_sub = 'sub-verse500'
sample_split = 'train'
raw_candidates = glob.glob(str(INPUT_ROOT / sample_split / 'rawdata' / sample_sub / '*_ct.nii.gz'))
msk_candidates = glob.glob(str(INPUT_ROOT / sample_split / 'derivatives' / sample_sub / '*_seg-vert_msk.nii.gz'))

if raw_candidates and msk_candidates:
    img_nii = nib.load(raw_candidates[0])
    msk_nii = nib.load(msk_candidates[0])
    im_iso = resample_nib(img_nii, (1, 1, 1), 3)
    ms_iso = resample_nib(msk_nii, (1, 1, 1), 0)
    im_np = reorient_to(im_iso, ('I', 'P', 'L')).get_fdata()
    ms_np = reorient_to(ms_iso, ('I', 'P', 'L')).get_fdata()
    areas = [np.sum(ms_np[:, :, i] > 0) for i in range(im_np.shape[2])]
    threshold = max(MIN_PIXELS, THRESHOLD_PERCENT * max(areas))
    kept = [i for i, area in enumerate(areas) if area >= threshold]
    vis_idx = kept[len(kept) // 2] if kept else im_np.shape[2] // 2
    raw_slice = im_np[:, :, vis_idx]

    baseline_bone = window_image(raw_slice, *WINDOWS['bone'])
    baseline_rgb = np.repeat(baseline_bone[:, :, None], 3, axis=2)
    mw = multi_window_image(raw_slice)

    fig, ax = plt.subplots(2, 4, figsize=(16, 9))

    ax[0, 0].imshow(baseline_bone, cmap='gray')
    ax[0, 0].set_title('Baseline C1: bone')
    ax[0, 1].imshow(baseline_bone, cmap='gray')
    ax[0, 1].set_title('Baseline C2: bone')
    ax[0, 2].imshow(baseline_bone, cmap='gray')
    ax[0, 2].set_title('Baseline C3: bone')
    ax[0, 3].imshow(baseline_rgb)
    ax[0, 3].set_title('Baseline channels: [bone, bone, bone]')

    ax[1, 0].imshow(mw[:, :, 0], cmap='gray')
    ax[1, 0].set_title('Multi-window C1: soft tissue')
    ax[1, 1].imshow(mw[:, :, 1], cmap='gray')
    ax[1, 1].set_title('Multi-window C2: bone')
    ax[1, 2].imshow(mw[:, :, 2], cmap='gray')
    ax[1, 2].set_title('Multi-window C3: dense bone')
    ax[1, 3].imshow(mw)
    ax[1, 3].set_title('Multi-window composite')

    for a in ax.ravel():
        a.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Sample subject not found. Skip visualization or update sample_sub/sample_split.')

## 4. Export Functions

In [ ]:
def get_paths_for_style(style, subject_id, split_name, base_output):
    style_root = Path(base_output) / style
    paths = {
        'img': style_root / split_name / 'images',
        'sem': style_root / split_name / 'annotations_semantic',
        'ins': style_root / split_name / 'annotations_instance',
        'pan': style_root / split_name / f'{style}_panoptic',
        'suffixes': {'img': '', 'sem': '', 'ins': '', 'pan': ''},
    }
    if style == 'cityscapes':
        paths['img'] = style_root / split_name / 'leftImg8bit' / subject_id
        paths['sem'] = style_root / split_name / 'gtFine' / subject_id
        paths['ins'] = style_root / split_name / 'gtFine' / subject_id
        paths['pan'] = style_root / split_name / 'cityscapes_panoptic'
        paths['suffixes'] = {'img': '_leftImg8bit', 'sem': '_gtFine_labelIds', 'ins': '_gtFine_instanceIds', 'pan': ''}
    elif style == 'mapillary':
        paths['sem'] = style_root / split_name / 'labels'
        paths['ins'] = style_root / split_name / 'instances'
        paths['pan'] = style_root / split_name / 'panoptic'
    return paths


def export_granular(sub_id, s_idx, img_rgb, inst_msk, sem_msk, style, split_name, base_output, selected_tasks):
    cfg = get_paths_for_style(style, sub_id, split_name, base_output)
    base_name = f'{sub_id}_sag_{s_idx:03d}'

    pan_img, segments = get_panoptic_encoding(inst_msk)
    sub_num = int(''.join(filter(str.isdigit, sub_id)))
    image_id = sub_num * 1000 + s_idx
    info = {
        'file_name': f"{base_name}{cfg['suffixes']['img']}.png",
        'id': image_id,
        'segments': segments,
        'width': img_rgb.shape[1],
        'height': img_rgb.shape[0],
    }

    cfg['img'].mkdir(parents=True, exist_ok=True)
    Image.fromarray(img_rgb).save(cfg['img'] / f"{base_name}{cfg['suffixes']['img']}.png")

    if 'semantic' in selected_tasks:
        cfg['sem'].mkdir(parents=True, exist_ok=True)
        Image.fromarray(sem_msk).save(cfg['sem'] / f"{base_name}{cfg['suffixes']['sem']}.png")

    if 'instance' in selected_tasks:
        cfg['ins'].mkdir(parents=True, exist_ok=True)
        save_ins = inst_msk.copy()
        if style == 'cityscapes':
            save_ins = save_ins.astype(np.uint32) * 1000 + 1
            save_ins[inst_msk == 0] = 0
        Image.fromarray(save_ins).save(cfg['ins'] / f"{base_name}{cfg['suffixes']['ins']}.png")

    if 'panoptic' in selected_tasks:
        cfg['pan'].mkdir(parents=True, exist_ok=True)
        Image.fromarray(pan_img).save(cfg['pan'] / f"{base_name}{cfg['suffixes']['pan']}.png")

    return info


def process_subject(sub_id, raw_path, msk_path, split_name, style, base_output, selected_tasks):
    try:
        im_iso = resample_nib(nib.load(raw_path), (1, 1, 1), 3)
        ms_iso = resample_nib(nib.load(msk_path), (1, 1, 1), 0)
        im_np = reorient_to(im_iso, ('I', 'P', 'L')).get_fdata()
        ms_np = reorient_to(ms_iso, ('I', 'P', 'L')).get_fdata()

        areas = [np.sum(ms_np[:, :, i] > 0) for i in range(im_np.shape[2])]
        if not areas or max(areas) <= 0:
            return []
        threshold = max(MIN_PIXELS, THRESHOLD_PERCENT * max(areas))
        kept = [i for i, area in enumerate(areas) if area >= threshold]

        all_info = []
        for i in kept:
            img_rgb = multi_window_image(im_np[:, :, i])
            inst_msk = ms_np[:, :, i].astype(np.uint8)
            sem_msk = np.vectorize(map_semantic_class)(inst_msk).astype(np.uint8)
            info = export_granular(sub_id, i, img_rgb, inst_msk, sem_msk, style, split_name, base_output, selected_tasks)
            all_info.append(info)
        return all_info
    except Exception as exc:
        print(f'Failed {sub_id}: {exc}')
        return []

## 5. Run Processing

This writes both `data/ade20k` and `data/coco`. If `FORCE_REPROCESS=False`, existing style folders are kept.

In [ ]:
categories = [{'id': i + 1, 'name': V_NAMES[i], 'isthing': 1} for i in range(len(V_NAMES))]
categories.append({'id': 0, 'name': 'background', 'isthing': 0})

all_processed_data = {}

for style in EXPORT_STYLES:
    selected_tasks = SELECTED_TASKS_BY_STYLE.get(style, [])
    if not selected_tasks:
        print(f'Skip style {style}: no selected tasks')
        continue

    style_root = OUTPUT_ROOT / style
    if FORCE_REPROCESS and style_root.exists():
        print('Removing existing output:', style_root)
        shutil.rmtree(style_root)

    style_processed = {}
    for split in SPLITS:
        split_root = style_root / split
        split_root.mkdir(parents=True, exist_ok=True)
        print()
        print(f'Processing split={split}, style={style}, tasks={selected_tasks}')

        subs = sorted(glob.glob(str(INPUT_ROOT / split / 'rawdata' / 'sub-*')))
        all_res = []
        for sub_dir in tqdm(subs, desc=f'{style}:{split}'):
            sid = os.path.basename(sub_dir)
            raw_files = glob.glob(str(Path(sub_dir) / '*_ct.nii.gz'))
            msk_files = glob.glob(str(INPUT_ROOT / split / 'derivatives' / sid / '*_seg-vert_msk.nii.gz'))
            if raw_files and msk_files:
                all_res.extend(process_subject(sid, raw_files[0], msk_files[0], split, style, OUTPUT_ROOT, selected_tasks))

        json_path = split_root / f'verse_{split}_metadata.json'
        images = [{'id': r['id'], 'file_name': r['file_name'], 'width': r['width'], 'height': r['height']} for r in all_res]
        annotations = [{'image_id': r['id'], 'file_name': r['file_name'], 'segments_info': r['segments']} for r in all_res]
        with open(json_path, 'w') as f:
            json.dump({'images': images, 'annotations': annotations, 'categories': categories}, f, indent=4)

        style_processed[split] = all_res
        print(f'Wrote {len(all_res)} slices to {split_root}')

    all_processed_data[style] = style_processed

## 6. Dataset Statistics and RGB Sanity Check

In [ ]:
for style, split_data in all_processed_data.items():
    print()
    print(f'Style: {style}')
    print(f"{'Split':<10} | {'Total slices':<15}")
    print('-' * 30)
    for split, data in split_data.items():
        print(f'{split:<10} | {len(data):<15}')

# Confirm one exported image is RGB and channels are not identical.
for style in EXPORT_STYLES:
    image_files = sorted((OUTPUT_ROOT / style).glob('*/images/*.png'))
    if image_files:
        arr = np.array(Image.open(image_files[0]))
        print()
        print('Sample:', image_files[0])
        print('Shape:', arr.shape, 'dtype:', arr.dtype)
        if arr.ndim == 3 and arr.shape[2] == 3:
            print('R==G:', bool(np.array_equal(arr[:, :, 0], arr[:, :, 1])))
            print('G==B:', bool(np.array_equal(arr[:, :, 1], arr[:, :, 2])))
        break